In [2]:
import requests

#get total cow genes info
url= 'https://rest.kegg.jp/find/genes/bta'
requestData = requests.get(url)
cow_gene_info=dict()
for line in requestData.text.split('\n'):
    try:
        organism, gene_number= line.split('\t')[0].split(':')
        inform= line.split('\t')[1]
        if organism=='bta':
            gene_name= inform.split(';')[0]
            cow_gene_info[gene_number]= gene_name
    except:
        pass

In [3]:
import re
pattern = r'K([0-9]*)'
cow_gene_k_number= dict()
#get cow genes K number for reaction extraction
for gene_number, gene_name in cow_gene_info.items():
    url= 'https://rest.kegg.jp/get/bta:'+gene_number
    requestdata= requests.get(url)
    for inform in requestdata.text.split('\n'):
        if 'ORTHOLOGY' in inform:
            a= re.search(pattern,inform)
            string= a.group()
            cow_gene_k_number[gene_name]= string
            break


In [4]:
#get reactions of corresponding K number
k_number_reaction= dict()
for gene_name, k_number in cow_gene_k_number.items():
    reaction_list=[]
    url= 'https://rest.kegg.jp/get/'+k_number
    requestdata= requests.get(url)
    reaction_start_index=False
    reaction_end_index= False
    for i in range(len(requestdata.text.split('\n'))):
        inform= requestdata.text.split('\n')[i]
        if 'REACTION' in inform:
            reaction_start_index=i
            continue
        if reaction_start_index:
            if not reaction_end_index:
                if inform[0] != ' ':
                    reaction_end_index=i
    
    if reaction_start_index:
        for index in range(reaction_start_index, reaction_end_index):
            reaction_inform= requestdata.text.split('\n')[index]
            reaction_name= reaction_inform.strip().replace('REACTION','').strip().split('  ')[0]
            reaction_list.append(reaction_name)
    if not len(reaction_list)==0:
        k_number_reaction[gene_name]= reaction_list

In [5]:
k_number_reaction['LOC534913']= k_number_reaction['mariner-like transposase']
del k_number_reaction['mariner-like transposase']

k_number_reaction['LOC618367']= k_number_reaction['uncharacterized protein LOC618367']
del k_number_reaction['uncharacterized protein LOC618367']

k_number_reaction['LOC521656']= k_number_reaction['cytochrome P450, family 2, subfamily J']
del k_number_reaction['cytochrome P450, family 2, subfamily J']

k_number_reaction['LOC100125266']= k_number_reaction['alkaline phosphatase-like precursor']
del k_number_reaction['alkaline phosphatase-like precursor']

k_number_reaction['LOC100125266']= k_number_reaction['carboxylesterase 1 precursor']
del k_number_reaction['carboxylesterase 1 precursor']

k_number_reaction['LOC539818']= k_number_reaction['Dolichyl-diphosphooligosaccharide--protein glycosyltransferase subunit 1-like precursor']
del k_number_reaction['Dolichyl-diphosphooligosaccharide--protein glycosyltransferase subunit 1-like precursor']

k_number_reaction['LOC527068']= k_number_reaction['aldo-keto reductase family 1 member C3 -like']
del k_number_reaction['aldo-keto reductase family 1 member C3 -like']

k_number_reaction['LOC615514']= k_number_reaction['glutathione S-transferase mu 1-like']
del k_number_reaction['glutathione S-transferase mu 1-like']

k_number_reaction['LOC538702']= k_number_reaction['glycerol kinase']
del k_number_reaction['glycerol kinase']

k_number_reaction['LOC781988']= k_number_reaction['uncharacterized protein LOC781988 precursor']
del k_number_reaction['uncharacterized protein LOC781988 precursor']

k_number_reaction['LOC518623']= k_number_reaction['uncharacterized protein LOC518623']
del k_number_reaction['uncharacterized protein LOC518623']

KeyError: 'uncharacterized protein LOC618367'

In [6]:
import pickle
with open('gene_reaction.pickle', 'wb') as f:
    pickle.dump(k_number_reaction,f)
with open('gene_k_number.pickle', 'wb') as f:
    pickle.dump(cow_gene_k_number,f)